### Chatbot And RAG Evaluation

Retrieval Augmented Generation (RAG) is a technique that enhances Large Language Models (LLMs) by providing them with relevant external knowledge. It has become one of the most widely used approaches for building LLM applications.

This tutorial will show you how to evaluate your RAG applications using LangSmith. You'll learn:

- How to create test datasets
- How to run your RAG application on those datasets
- How to measure your application's performance using different evaluation metrics
### Overview
A typical RAG evaluation workflow consists of three main steps:

- Creating a dataset with questions and their expected answers
- Running your RAG application on those questions
- Using evaluators to measure how well your application performed, looking at factors like:
   - Answer relevance
   - Answer accuracy
   - Retrieval quality

For this tutorial, we'll create and evaluate a bot that answers questions about a few of Lilian Weng's insightful blog posts.

### Chatbot Evaluation

In [55]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")
os.environ["LANGSMITH_TRACING"]="true"

In [57]:
### create the datapoints
from langsmith import Client
client=Client()

## Define the dataset-these are your test data

dataset_name="Chatbot Evaluation latest "
dataset=client.create_dataset(dataset_name)

client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }
    ]
)

{'example_ids': ['80e678bb-b90b-4836-94bc-20e46437d6ac',
  '74e67d03-7716-49f3-ba4a-e77206692a1f',
  '78182c5f-97a6-4667-b542-6a2c4b229ab9',
  'ef325270-4099-413c-b341-5bf7ae0861b2',
  'a0b1d9d9-e5af-46de-b5a6-b58098dab4f8'],
 'count': 5,
 'as_of': '2026-10-20T14:40:37.514044032Z'}

### Define Metrics(LLM As A Judge)

In [58]:
from langsmith import wrappers
from langchain_google_genai import ChatGoogleGenerativeAI

google_client = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

eval_instructions = "You are an expert professor specialized in grading students' answers to questions."

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    user_content = f"""You are grading the following question:
    {inputs['question']}
    Here is the real answer:
    {reference_outputs['answer']}
    You are grading the following predicted answer:
    {outputs['response']}
    Respond with CORRECT or INCORRECT:
    Grade:
    """

    response = google_client.invoke([
        ("system",eval_instructions),
        ("human",user_content)
    ]).content
    
    if isinstance(response, list):
        response = "".join(
            item.get("text", "") if isinstance(item, dict) else str(item)
            for item in response
        )
    
    return response.strip() == "CORRECT"

In [59]:
## Concisions- checks whether the actual output is less than 2x the length of the expected result.

def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

### Run Evaluations

In [60]:
default_instructions = "Respond to the users question in a short, concise manner (one short sentence)."
def my_app(question: str, model: str = "gemini-3.5-live-translate-preview", instructions: str = default_instructions) -> str:
    response=google_client.invoke([
        ("system",instructions),
        ("human",question)
    ])
    return response.content

In [61]:
### Call my_app for every datapoints
def ls_target(inputs: dict) -> dict:
    return {"response": my_app(inputs["question"])}

In [62]:
## Run our evaluation
dataset_name="Chatbot Evaluation latest"
expeirment_results=client.evaluate(
    ls_target,##Your AI System
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="gemini-3.5-live-translate-preview"
)

View the evaluation results for experiment: 'gemini-3.5-live-translate-preview-ca3941ae' at:
https://smith.langchain.com/o/2797c537-3141-4d0c-9ad9-bf67f1b9e862/datasets/6c3543a2-4a0c-4da2-9f83-12929d569b20/compare?selectedSessions=630f86ef-d1ac-4e28-a3bd-a179a69e38c8




5it [04:48, 57.78s/it]
